# MEK oscillation IHC analysis

This notebook measures Her7-Venus, pMEK and ppERK along the PSM of each embryo and compares the profiles between oscillation phases. The image processing and analysis code is in `ihc_pipeline.py`. The bands are drawn in Fiji with `IHC_ROI_only.ijm`.

The sections follow the order of the analysis. The samples are found first, then the bands are prepared, the masks are made and the samples are measured, and finally the profiles are built and compared between phases. Running a single sample first and checking its QC figure makes it easier to catch problems before all samples are measured.

The notebook reads the image files without changing them and saves everything it produces in the output folder.

## 1. Setup

The environment is installed once with `uv sync --locked`, adding `--extra cuda` on computers with an NVIDIA GPU, and the project's `.venv` is then selected as the kernel. The README describes this in more detail. The next cell assumes the kernel starts in the `analysis_scripts` folder. If it starts somewhere else, set `ANALYSIS_DIR` to that folder.

All settings are in the cell after that. The `settings` dictionary holds the ones that are changed most often, and `ADVANCED` can set any other setting by its name. The table below that cell lists every setting.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

ANALYSIS_DIR = Path.cwd()
if not (ANALYSIS_DIR / "ihc_pipeline.py").exists():
    raise FileNotFoundError("Set ANALYSIS_DIR to the project's analysis_scripts directory.")
sys.path.insert(0, str(ANALYSIS_DIR))
import ihc_pipeline as ihc

In [ ]:
# -------- Files and sample selection --------
# The folder that holds the replicate folders (N_1..., N_2...) and the folder for
# results. The environment variables, when set, replace these paths.
EXPERIMENTAL_DATA = Path(os.environ.get(
    "IHC_EXPERIMENTAL_DATA", str(ANALYSIS_DIR.parent / "Exp_1__mek_h7v_pperk")))
OUTPUT_ROOT = Path(os.environ.get("IHC_OUTPUT_ROOT", str(ANALYSIS_DIR / "ihc_results")))

# Optional table of distance ranges to leave out (see section 6). It is used only if
# the file exists.
EXCLUSIONS = ANALYSIS_DIR / "IHC_exclusions.csv"

# Samples to run, as "replicate/sample_id". Empty runs every sample that has ROIs.
SAMPLE_KEYS = []

# -------- What to run --------
RUN_MEASUREMENTS = False     # make masks and measure the selected samples
RELOAD_SAVED = True         # load the saved measurements when nothing is measured
PLOT_REPLICATES_SEPARATELY = True# also draw the phase plots for each replicate on its own
ALIGN_PROFILES = 1         # 0 aligns posterior ends (posterior to anterior), 1 aligns anterior ends (anterior to posterior)

# -------- Settings (the table below lists all of them) --------
settings = dict(
    experimental_data=EXPERIMENTAL_DATA,
    output_root=OUTPUT_ROOT,
    replicates=(),                    # e.g. ("N_1", "N_2"), or empty for all
    nis_preference="nd2",             # denoised ND2 or TIFF, when both exist
    use_ai_denoise=True,              # segment the NIS-denoised DAPI
    use_cellpose=True,                # False thresholds the DAPI instead
    mask_source="compute",           # compute, existing, or edited (corrected in Fiji)
    width_override_px=None,          # None uses each ROI's own width
    combine_sides=False,             # True combines an embryo's sides
    align_profiles=ALIGN_PROFILES,   # set by ALIGN_PROFILES above
    analysis_method="per_slice",     # per_slice, pooled, block_mip or full_mip
    z_average="pool",                # pool, mean or median over Z
    background_subtract=True,
    min_mask_fraction=0.05,          # blank the mean where a compartment covers < 5% of the band
    protein_median_size=3,           # 0 turns the median filter off
    protein_outliers=True,           # replace bright outlier pixels
    mask_cap=4095.0,                 # None turns the cap off
    exclusions_csv=str(EXCLUSIONS if EXCLUSIONS.exists() else ""),  # ranges to leave out
    device="auto",                  # CUDA, then Apple MPS, then CPU
    image_workers=6,                 # samples measured in parallel
    threads_per_worker=1,
    phase_files={},                  # e.g. {"N_1": "/path/phases.csv"}
    phase_overrides={},              # e.g. {"N_2/triple_ihc_014_P1": "2"}
)

# Any other setting, by its name in the table below.
ADVANCED = {
    # "cellprob_threshold": 0.0,
    # "flow_threshold": 0.4,
    # "cellpose_diameter_px": None,
    # "normalization_window_um": 40.0,
    # "smooth_average": True,          # smooths the plotted average only
    "phase_colormap": "Dark2",
    # --- baseline and day normalization ---
    "background_percentile": 5.0,          # baseline = this percentile of the smoothed profiles
    "normalization_window_um": 100.0,       # rolling-median width used to find the baseline
    "day_normalize": False,                 # divide each imaging day by its median, per channel and compartment
    "day_reference_um": (0.0, 400.0),      # distance range (µm) that sets each day's median

    # --- Savitzky–Golay smoothing of the plotted average ---
    "smooth_average": True,                # False turns smoothing off
    "savgol_window_um":40.0,              # window width (µm)
    "savgol_order": 1,                     # polynomial order
    "interpolation": "nearest",
    "mip_block_size": 5,
    "phase_minimum_lines": 2,
}
settings.update(ADVANCED)
cfg = ihc.Config(**settings).validate()

### Settings reference

This table lists every setting with its current value, its allowed choices and a description. A value of `None` either turns a setting off or means that the value is read from the image, and the description of each setting says which applies.

With `use_ai_denoise=True`, the DAPI of the NIS-denoised export is segmented directly, without the filtering that is applied to raw DAPI.

In [ ]:
parameters = ihc.parameter_table(cfg)
with pd.option_context("display.max_colwidth", 140, "display.max_rows", None):
    display(parameters)

## 2. Samples, NIS pairing and phases

Each row below is one sample, which is either an exported TIFF or one position of an ND2 file. When a position exists in both forms, the protein channels are read from the TIFF. `nis_path` shows the NIS-denoised file whose DAPI will be segmented, and when both a denoised ND2 and a denoised TIFF exist, `nis_preference` decides between them.

Phases are read from `phases.csv` in each replicate folder, or from the file named in `phase_files`. The table can have the columns `file,phase` or `sample_id,phase`, and `phase_source` shows which file each phase came from. Samples without a phase are measured like the others and are left out of the phase averages.

In [ ]:
samples = ihc.discover_samples(cfg)
samples["sample_key"] = samples.replicate + "/" + samples.sample_id
samples["has_nis"] = samples.nis_path.ne("")
display(samples[["sample_key", "has_rois", "has_nis", "phase", "phase_source", "raw_path", "nis_path"]])
display(samples.groupby("replicate").agg(
    images=("sample_id", "size"), ready_rois=("has_rois", "sum"), nis_pairs=("has_nis", "sum")))

selected = samples[samples.has_rois].copy()
if SAMPLE_KEYS:
    missing = set(SAMPLE_KEYS) - set(samples.sample_key)
    if missing:
        raise ValueError(f"Selected keys were not found: {sorted(missing)}")
    selected = selected[selected.sample_key.isin(SAMPLE_KEYS)].copy()
print(f"Selected {len(selected)} images with ROIs. {sum(~samples.has_rois)} images do not have ROIs yet.")

## 3. Bands (ROIs)

All the bands of an image are saved in one file, `output_root/rois/<replicate>/<sample>/rois.csv`. Each row is a clicked point (`roi_id, vertex_index, x_px, y_px, width_px`) in image pixels counted from 0, and the first point of each side is at its posterior end. `width_px` sets the width of the measured band, which can differ from the thickness of the line drawn on screen.

The bands are drawn in Fiji with `IHC_ROI_only.ijm`, which asks whether the images are ND2 files or exported TIFFs and saves each image's bands in this format. After drawing new bands, run the inventory cell in section 2 again so the new samples show `has_rois = True`.

The next cell converts older ImageJ ROI ZIPs into this format. CSV files that already exist are kept as they are, so edits made in Fiji are preserved when the cell is run again.

A nuclear mask can be corrected in Fiji. Open the computed `Mask.tif` from the `masks_...` folder, paint the corrections, and save the result as `edited_masks/<replicate>/<sample>/Mask.tif` in the output folder. These masks are used for the measurement when `mask_source="edited"`.

In [ ]:
conversion = ihc.convert_rois(samples[samples.has_rois], cfg)
display(conversion)

## 4. Masks and measurements

With `use_ai_denoise=True`, the DAPI of each sample's NIS-denoised export is segmented. With `use_ai_denoise=False`, the raw DAPI is first median filtered, background subtracted and contrast enhanced. With `use_cellpose=False`, each Z-plane is thresholded instead of being segmented by Cellpose.

All Z methods are measured from the same filtered protein images, masks and bands. The cytoplasmic mask covers everything outside the nuclei, and the maximum projections are taken after masking. With 31 planes, the default blocks are 2–4, 5–7, …, 29–31.

Before the measurement, the protein channels are median filtered, bright outlier pixels are replaced and values inside the masks are capped at 4095. Each of these steps can be switched off in the settings.

The measurements of each sample are saved as a Parquet file, and `export_csv=True` also writes a CSV. Masks are saved in a folder named after the way they were made, such as `masks_cellpose_denoised` or `masks_threshold_raw`, which keeps masks made with different `use_cellpose` or `use_ai_denoise` settings apart. Saved masks are reused while `reuse_cache` is True. After changing another mask setting, such as a Cellpose threshold, set `reuse_cache` to False so that the masks are made again.

In [ ]:
measurements = pd.DataFrame()
if RUN_MEASUREMENTS:
    if selected.empty:
        raise ValueError("No selected sample has ROIs. Draw or convert ROIs first.")
    measurements = ihc.run(selected, cfg)
    print("Run metadata:", measurements.attrs.get("run_directory"))
elif RELOAD_SAVED:
    measurements = ihc.load_measurements(selected, cfg)
    print("Loaded the saved measurements.")
else:
    print("Nothing was measured or loaded. Set RUN_MEASUREMENTS or RELOAD_SAVED to True.")
if not measurements.empty:
    display(measurements.head())
    display(measurements.groupby(["replicate", "sample_id", "method"]).size().rename("rows"))

### What a profile contains

At each position, the band has many points across its width, and `mask_weight / band_weight` is the fraction of them inside the compartment. Because the mask is interpolated, a point at the edge of a nucleus can count partly.

- `masked_mean = masked_sum / mask_weight` is the mean intensity inside the compartment.
- `band_mean = masked_sum / band_weight` is the compartment's signal per band area.
- Where the compartment covers less than `min_mask_fraction` of the band, `masked_mean` is left empty. The sums are kept, and pooling over Z still uses them.

A pooled profile adds the sums of all planes before dividing, which weights each plane by how much of the compartment it contains. Averaging the means of the individual planes would instead give every plane the same weight.

## 5. Check a mask

The figure shows the raw DAPI of the first selected sample. Next to it is the same plane with the nuclear mask in green, the measured bands in blue and the posterior start of each band in red. The mask is the same one used for the measurements. Looking at a few Z-planes, with the `z` argument of `qc_figure`, shows whether a mask made from NIS-denoised DAPI lines up with the raw image.

In [ ]:
if not measurements.empty:
    figure = ihc.qc_figure(selected.iloc[0], cfg)
    display(figure)
    plt.close(figure)

## 6. Profiles

Ranges listed in the exclusions CSV are left out at this step. Its columns are `replicate,sample_id,side,start_um,stop_um,channels,reason`, and `all` in `channels` applies a range to every channel. The exclusions are applied to the profiles, and the saved measurements stay as they are.

Each embryo's baseline is a low percentile (`background_percentile`) of its profiles after they are smoothed with a rolling median (`normalization_window_um`). The baseline is shared by all sides of the embryo, calculated separately for each channel and compartment, and subtracted from the unsmoothed profiles. With `day_normalize=True`, the profiles of each imaging day are then divided by that day's median within `day_reference_um`.

By default each side is a separate profile, and `combine_sides=True` combines the sides of an embryo into one. Each profile ends where its band ends. Distances are the position along the band multiplied by the pixel size.

In [ ]:
profiles = normalization = pd.DataFrame()
if not measurements.empty:
    profiles, normalization = ihc.side_profiles(measurements, cfg)
    display(normalization)
    display(profiles.head())
    individual_figure = ihc.plot_individual_profiles(profiles, cfg)
    display(individual_figure)
    plt.close(individual_figure)
    bad = normalization[normalization.status.ne("ok")]
    if len(bad):
        print("These embryos have no baseline because their profiles are empty:")
        display(bad)

## 7. Phase summaries

Phases are taken from the annotation files. Samples whose labels are not listed in `phase_labels` are shown with the individual profiles and left out of the phase plots.

With sides kept separate, `n` counts sides, and `n_embryos` and `n_replicates` are given alongside it. With `combine_sides=True`, `n` counts embryos. `n` decreases where profiles end or ranges are excluded, and an average is drawn only where `n` reaches `phase_minimum_lines`. When `smooth_average` is True the plotted average is smoothed, while the saved statistics and error bands are calculated from the unsmoothed profiles.

In [ ]:
phase_summary = overall_summary = pd.DataFrame()
if not profiles.empty:
    phase_summary = ihc.summarize(profiles, cfg, by_phase=True)
    overall_summary = ihc.summarize(profiles, cfg, by_phase=False)
    display(profiles.groupby(["replicate", "phase"]).profile_unit.nunique().rename("profile_units"))
    display(phase_summary.head())
    # Phase plots of each replicate on its own, shown below the combined plot.
    replicate_figures = {}
    if PLOT_REPLICATES_SEPARATELY:
        for replicate, rep_profiles in profiles.groupby("replicate", sort=True, observed=True):
            rep_summary = ihc.summarize(rep_profiles, cfg, by_phase=True)
            for channel, figure in ihc.plot_phases(rep_profiles, cfg, rep_summary).items():
                figure.suptitle(f"{replicate}: {figure.get_suptitle()}",
                                x=0.01, ha="left", color=ihc.INK, fontsize=22)
                replicate_figures.setdefault(channel, []).append(figure)
    for channel, figure in ihc.plot_phases(profiles, cfg, phase_summary).items():
        for shown in [figure] + replicate_figures.get(channel, []):
            display(shown)
            plt.close(shown)
    for channel, figure in ihc.plot_phase_panels(profiles, cfg, phase_summary).items():
        display(figure)
        plt.close(figure)

## 8. Save the analysis

This cell saves the profiles, baselines and day factors, the phase and overall summaries, the settings, the package versions and the figures in a new folder under `analysis/`.

To change only the normalization, the side combination or the plots, reload the saved measurements and run sections 6 to 8 again. If the phase annotations change, run section 2 again before reloading. Changes to the masks, bands, protein filtering or Z settings require running section 4 again.

In [ ]:
if not profiles.empty:
    analysis_dir = ihc.export_analysis(profiles, normalization, phase_summary, cfg)
    overall_summary.to_csv(analysis_dir / "overall_summary.csv", index=False)
    individual_figure = ihc.plot_individual_profiles(profiles, cfg)
    individual_figure.savefig(analysis_dir / "individual_profiles.png", dpi=cfg.dpi)
    plt.close(individual_figure)
    for channel, figure in ihc.plot_phases(profiles, cfg, phase_summary).items():
        figure.savefig(analysis_dir / f"{channel}_phases.png", dpi=cfg.dpi)
        plt.close(figure)
    print("Saved:", analysis_dir)

## 9. Further exploration

`measurements`, `profiles`, `normalization`, `phase_summary` and `overall_summary` are ordinary pandas tables that can be explored here.